# Step 6 — Load Segments + Risk/Value into MySQL, Export for Power BI

**Input:** `data/segments.csv`, `data/risk_value_segments.csv`
**Output:** `telco_churn.segments`, `telco_churn.risk_value` tables,
the `vw_dashboard` view, and `data/powerbi_dashboard_data.csv`
(a flattened export used as the Power BI data source).

Run after `load_to_mysql.ipynb` (customers table must already exist),
`segmentation.ipynb`, and `churn_clv_model.ipynb`.

In [1]:
import os
import sys


def find_project_root(marker="requirements.txt"):
    path = os.getcwd()
    for _ in range(5):
        if os.path.exists(os.path.join(path, marker)):
            return path
        path = os.path.dirname(path)
    raise FileNotFoundError(f"Could not locate project root (looked for {marker})")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, os.path.join(PROJECT_ROOT, "scripts"))
print(f"PROJECT_ROOT = {PROJECT_ROOT}")

PROJECT_ROOT = C:\Users\DEVKUMAR\OneDrive\Desktop\DA_Project


In [2]:
import pandas as pd

from utils_db import get_connection

SCHEMA_PATH = os.path.join(PROJECT_ROOT, "sql", "03_powerbi_schema.sql")
SEGMENTS_CSV = os.path.join(PROJECT_ROOT, "data", "segments.csv")
RISK_VALUE_CSV = os.path.join(PROJECT_ROOT, "data", "risk_value_segments.csv")
DASHBOARD_CSV = os.path.join(PROJECT_ROOT, "data", "powerbi_dashboard_data.csv")

SEGMENTS_COLS = ["customerID", "R_score", "F_score", "M_score", "RFM_Total", "Segment"]
RISK_VALUE_COLS = ["customerID", "Historical_CLV", "Projected_CLV", "Churn_Risk_Score", "Risk_Value_Quadrant"]

## Create `segments` / `risk_value` tables + `vw_dashboard` view

In [3]:
conn = get_connection()
cur = conn.cursor()
with open(SCHEMA_PATH, encoding="utf-8") as f:
    for statement in f.read().split(";"):
        statement = statement.strip()
        if statement:
            cur.execute(statement)
conn.commit()
cur.close()
print("Schema created.")

Schema created.


## Load `segments`

In [4]:
def load_table(conn, csv_path, table, cols):
    df = pd.read_csv(csv_path)[cols]
    cur = conn.cursor()
    cur.execute(f"TRUNCATE TABLE {table}")
    placeholders = ", ".join(["%s"] * len(cols))
    insert_sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({placeholders})"
    rows = [tuple(r) for r in df.itertuples(index=False, name=None)]
    cur.executemany(insert_sql, rows)
    conn.commit()
    cur.execute(f"SELECT COUNT(*) FROM {table}")
    count = cur.fetchone()[0]
    cur.close()
    print(f"Loaded {count} rows into {table} (source had {len(df)})")
    assert count == len(df)


load_table(conn, SEGMENTS_CSV, "segments", SEGMENTS_COLS)

Loaded 7043 rows into segments (source had 7043)


## Load `risk_value`

In [5]:
load_table(conn, RISK_VALUE_CSV, "risk_value", RISK_VALUE_COLS)

Loaded 7043 rows into risk_value (source had 7043)


## Export the flattened dashboard CSV for Power BI

In [6]:
cur = conn.cursor()
cur.execute("SELECT * FROM vw_dashboard")
cols = [d[0] for d in cur.description]
rows = cur.fetchall()
cur.close()
conn.close()

dashboard_df = pd.DataFrame(rows, columns=cols)
dashboard_df.to_csv(DASHBOARD_CSV, index=False)
print(f"Exported {len(dashboard_df)} rows to {DASHBOARD_CSV}")
dashboard_df.head()

Exported 7043 rows to C:\Users\DEVKUMAR\OneDrive\Desktop\DA_Project\data\powerbi_dashboard_data.csv


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,TenureBucket,PhoneService,MultipleLines,InternetService,...,ChurnFlag,R_score,F_score,M_score,RFM_Total,Segment,Historical_CLV,Projected_CLV,Churn_Risk_Score,Risk_Value_Quadrant
0,0002-ORFBO,Female,No,Yes,Yes,9,7-12 mo,Yes,No,DSL,...,0,2,3,3,8,Growing / Promising,593.30,2761.04,0.3293,High Value / Low Risk (protect & reward)
1,0003-MKNFE,Male,No,No,No,9,7-12 mo,Yes,Yes,DSL,...,0,2,2,3,7,New / Low-Engagement,542.40,1083.70,0.5822,Low Value / High Risk (low priority)
2,0004-TLHLJ,Male,No,No,No,4,0-6 mo,Yes,No,Fiber optic,...,1,1,2,3,6,New / Low-Engagement,280.85,1318.16,0.8755,Low Value / High Risk (low priority)
3,0011-IGKFF,Male,Yes,Yes,No,13,13-24 mo,Yes,No,Fiber optic,...,1,2,4,5,11,Growing / Promising,1237.85,1731.44,0.8480,High Value / High Risk (priority retention)
4,0013-EXCHZ,Female,Yes,Yes,No,3,0-6 mo,Yes,No,Fiber optic,...,1,1,2,4,7,Price-Sensitive Risk,267.40,1528.97,0.8556,Low Value / High Risk (low priority)
